In [2]:
import torchvision
# For image transforms
from torchvision import transforms
# For DATA SET
import torchvision.datasets as datasets
# For Pytorch methods
import torch
import torch.nn as nn
# For Optimizer
import torch.optim as optim
# FOR DATA LOADER
from torch.utils.data import DataLoader
# FOR TENSOR BOARD VISUALIZATION
from torch.utils.tensorboard import SummaryWriter # to print to tensorboard

In [9]:
# Hyperparameters
device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
lr = 3e-4
batchSize = 32  # Batch size
numEpochs = 100
logStep = 625  # the number of steps to log the images and losses to tensorboard

latent_dimension = 128 # 64, 128, 256
# for simplicity we will flatten the image to a vector and to use simple MLP networks
# 28 * 28 * 1 flattens to 784
# you are also free to use CNNs
image_dimension = 28 * 28 * 1  # 784

# we define a tranform that converts the image to tensor and normalizes it with mean and std of 0.5
# which will convert the image range from [0, 1] to [-1, 1]
myTransforms = transforms.Compose([transforms.ToTensor(),transforms.Normalize((0.5,), (0.5,))])

# the MNIST dataset is available through torchvision.datasets
print("loading MNIST digits dataset")
dataset = datasets.MNIST(root="dataset/", transform=myTransforms, download=True)
# let's create a dataloader to load the data in batches
loader = DataLoader(dataset, batch_size=batchSize, shuffle=True)


loading MNIST digits dataset


In [ ]:
class Generator(nn.Module):
    """
    Generator Model
    """
    def __init__(self):
        super().__init__()
        self.gen = nn.Sequential(
            nn.Linear(latent_dimension, 256),
            nn.BatchNorm1d(256), # Added Batch Norm
            nn.LeakyReLU(0.2),
            
            nn.Linear(256, 512),
            nn.BatchNorm1d(512), 
            nn.LeakyReLU(0.2),
            
            nn.Linear(512, 1024),
            nn.BatchNorm1d(1024), 
            nn.LeakyReLU(0.2),
            
            nn.Linear(1024, image_dimension),
            nn.Tanh(),  # It is helpful to use the tanh activation function to force the ouput into the [-1,1] range that our normalized images have.
        )


    def forward(self, x):
        return self.gen(x)



class Discriminator(nn.Module):
    """
    Discriminator Model
    """
    def __init__(self):
        super().__init__()
        self.disc = nn.Sequential(
            nn.Linear(image_dimension, 512), 
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            
            nn.Linear(256, 1),
            nn.Sigmoid()
        )


    def forward(self, x):
        return self.disc(x)

In [11]:
# initialize networks and optimizers
discriminator = Discriminator().to(device)
generator = Generator().to(device)
opt_discriminator = optim.Adam(discriminator.parameters(), lr=lr)
opt_generator = optim.Adam(generator.parameters(), lr=lr)

# This is a binary classification task, so we use Binary Cross Entropy Loss
criterion = nn.BCELoss()


In [12]:
import os
# Create images folder to save sample images as requested
os.makedirs("images", exist_ok=True)
# Set up fixed noise and Tensorboard SummaryWriter
fixed_noise = torch.randn(batchSize, latent_dimension).to(device)
writer = SummaryWriter(log_dir="runs/GAN_MNIST")

# Training Loop
step = 0
print("Started Training and visualization...")
for epoch in range(numEpochs):
    # loop over batches
    print()
    for batch_idx, (real, _) in enumerate(loader):
        # First we train the discriminator on real images vs. generated images

        # Get the real images and flatten them
        # for simplicity, we flatten the image to a vector and to use simple MLP networks
        # 28 * 28 * 1 flattens to 784
        real = real.view(-1, 784).to(device)
        batch_size = real.shape[0]

        # Step 1) generate fake images
        noise = torch.randn(batch_size, latent_dimension).to(device)
        fake = generator(noise)

        # Step 2) Train Discriminator:
        disc_real = discriminator(real).view(-1)
        loss_disc_real = criterion(disc_real, torch.ones_like(disc_real) * 0.9)

        disc_fake = discriminator(fake.detach()).view(-1)
        loss_disc_fake = criterion(disc_fake, torch.zeros_like(disc_fake))
        
        loss_discriminator = (loss_disc_real + loss_disc_fake) / 2
        
        discriminator.zero_grad()
        loss_discriminator.backward(retain_graph=True) # Retaining graph as per the hint
        opt_discriminator.step()
        # - predict the discriminator output for real images
        # - real images are labeled as 1
        # - calculate the loss for real images

        # - predict the discriminator output for fake images
        # -fake images are labeled as 0
        # -calculate the loss for fake images

        # -average the loss for real and fake images

        # - now upadate the weights of the discriminator by backpropagating the loss through the discriminator
        # the generator is not updated in this step
        # HINT: call the `backward` method of the discriminator with the argument `retain_graph=True` to keep the computational graph
        # this is necessary because we will use the same discriminator to train the generator

        # Train Generator:
        output = discriminator(fake).view(-1)
        loss_generator = criterion(output, torch.ones_like(output))
        generator.zero_grad()
        loss_generator.backward()
        opt_generator.step()
        # Now train the generator by generating fake images and passing them through the discriminator
        # You can do a little trick and modify the original objective function of
        # "minimizing the probability of the discriminator predicting the fake images as fake"
        # to "maximizing the probability of the discriminator predicting the fake images as real"
        # this leads to a faster training of the generator when it does not represent the real data well
        # this is a common trick in GANs
        # for moer information see section 17.1.2 of the book Deep Learning by Bishop and Bishop

        # Todo:
        # - pass the fake images through the discriminator
        # - calculate the loss (by passing the output of the discriminator through the criterion with labels set to 1 (real images
        # - update the weights of the generator


        # print the progress
        print(f"\rEpoch [{epoch}/{numEpochs}] Batch {batch_idx}/{len(loader)} \ Loss discriminator: {loss_discriminator:.4f}, loss generator: {loss_generator:.4f}", end="")

        # Log the losses and example images to tensorboard
        if batch_idx % logStep == 0:
            with torch.no_grad():
                # Generate noise via Generator, we always use the same noise to see the progression
                fake = generator(fixed_noise).reshape(-1, 1, 28, 28)
                # Get real data
                data = real.reshape(-1, 1, 28, 28)
                # make grid of pictures and add to tensorboard
                imgGridFake = torchvision.utils.make_grid(fake, normalize=True)
                imgGridReal = torchvision.utils.make_grid(data, normalize=True)

                # TODO: add the images and losses to tensorboard
                # HINT: use the SummaryWriter to add the images and scalars to tensorboard
                # HINT: use the `add_image` method to add the images to tensorboard
                # HINT: use the `add_scalar` method to add the losses to tensorboard
                writer.add_image("Fake_Images", imgGridFake, global_step=step)
                writer.add_image("Real_Images", imgGridReal, global_step=step)
                writer.add_scalar("Loss/Discriminator", loss_discriminator.item(), global_step=step)
                writer.add_scalar("Loss/Generator", loss_generator.item(), global_step=step)

                # Save generated images to images folder to see quality improvement
                torchvision.utils.save_image(fake, f"images/fake_images_step_{step}.png", normalize=True)

                # increment step
                step += 1

<>:75: SyntaxWarning: invalid escape sequence '\ '
<>:75: SyntaxWarning: invalid escape sequence '\ '
/var/folders/4m/dpkc51pn50g5mtmjpw8dsqj80000gn/T/ipykernel_36367/4042797324.py:75: SyntaxWarning: invalid escape sequence '\ '
  print(f"\rEpoch [{epoch}/{numEpochs}] Batch {batch_idx}/{len(loader)} \ Loss discriminator: {loss_discriminator:.4f}, loss generator: {loss_generator:.4f}", end="")


Started Training and visualization...

Epoch [0/100] Batch 1874/1875 \ Loss discriminator: 0.2580, loss generator: 5.80299
Epoch [1/100] Batch 1874/1875 \ Loss discriminator: 0.2732, loss generator: 4.89450
Epoch [2/100] Batch 1874/1875 \ Loss discriminator: 0.4377, loss generator: 1.9918
Epoch [3/100] Batch 1874/1875 \ Loss discriminator: 0.4946, loss generator: 2.2875
Epoch [4/100] Batch 1874/1875 \ Loss discriminator: 0.4278, loss generator: 1.7949
Epoch [5/100] Batch 1874/1875 \ Loss discriminator: 0.5109, loss generator: 2.0079
Epoch [6/100] Batch 1874/1875 \ Loss discriminator: 0.4966, loss generator: 1.3378
Epoch [7/100] Batch 1874/1875 \ Loss discriminator: 0.5899, loss generator: 1.3397
Epoch [8/100] Batch 1874/1875 \ Loss discriminator: 0.5613, loss generator: 1.5168
Epoch [9/100] Batch 1874/1875 \ Loss discriminator: 0.5159, loss generator: 1.3081
Epoch [10/100] Batch 1874/1875 \ Loss discriminator: 0.5659, loss generator: 1.3035
Epoch [11/100] Batch 1874/1875 \ Loss discrim

## Optional part

In [16]:
# --- WGAN Setup ---
class Critic(nn.Module):
    def __init__(self):
        super().__init__()
        self.crit = nn.Sequential(
            nn.Linear(image_dimension, 512),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(256, 1)
            # Notice there is NO Sigmoid activation here!
        )

    def forward(self, x):
        return self.crit(x)

# Initialize Critic
critic = Critic().to(device)

# WGAN typically uses RMSprop with a smaller learning rate instead of Adam
lr_wgan = 5e-5
opt_critic = optim.Adam(critic.parameters(), lr=lr_wgan)

# We reuse the same Generator architecture from earlier
generator_wgan = Generator().to(device)
opt_generator_wgan = optim.Adam(generator_wgan.parameters(), lr=lr_wgan)

# WGAN Hyperparameters
CRITIC_ITERATIONS = 5  # Train critic 5 times per generator update
WEIGHT_CLIP = 0.01     # Clip weights to [-0.01, 0.01]

writer_wgan = SummaryWriter(log_dir="runs/WGAN_MNIST")
step_wgan = 0


In [17]:
# --- WGAN Training Loop ---
print("Started WGAN Training...")
for epoch in range(numEpochs):
    for batch_idx, (real, _) in enumerate(loader):
        real = real.view(-1, 784).to(device)
        batch_size = real.shape[0]

        # ---------------------
        # Train Critic
        # ---------------------
        # We train the Critic multiple times before updating the Generator
        for _ in range(CRITIC_ITERATIONS):
            noise = torch.randn(batch_size, latent_dimension).to(device)
            fake = generator_wgan(noise)
            
            crit_real = critic(real).view(-1)
            crit_fake = critic(fake.detach()).view(-1)
            
            # WGAN Critic Loss: -(E[Critic(real)] - E[Critic(fake)])
            loss_critic = -(torch.mean(crit_real) - torch.mean(crit_fake))
            
            critic.zero_grad()
            loss_critic.backward()
            opt_critic.step()
            
            # Enforce 1-Lipschitz constraint by clipping Critic's weights
            for p in critic.parameters():
                p.data.clamp_(-WEIGHT_CLIP, WEIGHT_CLIP)

        # ---------------------
        # Train Generator
        # ---------------------
        # Generate a new batch of fake images to train the Generator
        noise = torch.randn(batch_size, latent_dimension).to(device)
        fake = generator_wgan(noise)
        output = critic(fake).view(-1)
        
        # WGAN Generator Loss: -E[Critic(fake)]
        loss_generator = -torch.mean(output)
        
        generator_wgan.zero_grad()
        loss_generator.backward()
        opt_generator_wgan.step()

        # ---------------------
        # Logging & Images
        # ---------------------
        if batch_idx % logStep == 0:
            print(f"\rEpoch [{epoch}/{numEpochs}] Batch {batch_idx}/{len(loader)} \ Loss Critic: {loss_critic:.4f}, Loss Generator: {loss_generator:.4f}", end="")
            
            with torch.no_grad():
                # Make sure to pass fixed_noise through our newly initialized WGAN Generator
                fake_logging = generator_wgan(fixed_noise).reshape(-1, 1, 28, 28)
                data_logging = real.reshape(-1, 1, 28, 28)
                
                imgGridFake = torchvision.utils.make_grid(fake_logging, normalize=True)
                imgGridReal = torchvision.utils.make_grid(data_logging, normalize=True)

                writer_wgan.add_image("WGAN Fake Images", imgGridFake, global_step=step_wgan)
                writer_wgan.add_image("WGAN Real Images", imgGridReal, global_step=step_wgan)
                writer_wgan.add_scalar("WGAN Loss/Critic", loss_critic.item(), global_step=step_wgan)
                writer_wgan.add_scalar("WGAN Loss/Generator", loss_generator.item(), global_step=step_wgan)
                
                # Save WGAN specific images to the folder
                torchvision.utils.save_image(fake_logging, f"images/wgan_fake_images_step_{step_wgan}.png", normalize=True)

                step_wgan += 1


<>:49: SyntaxWarning: invalid escape sequence '\ '
<>:49: SyntaxWarning: invalid escape sequence '\ '
/var/folders/4m/dpkc51pn50g5mtmjpw8dsqj80000gn/T/ipykernel_36367/3350728510.py:49: SyntaxWarning: invalid escape sequence '\ '
  print(f"\rEpoch [{epoch}/{numEpochs}] Batch {batch_idx}/{len(loader)} \ Loss Critic: {loss_critic:.4f}, Loss Generator: {loss_generator:.4f}", end="")


Started WGAN Training...
Epoch [99/100] Batch 1250/1875 \ Loss Critic: -0.3358, Loss Generator: -3.5186